In [1]:
import sys, platform, importlib.metadata, json
print("AUDIT_KERNEL="+json.dumps({"python":sys.version,"executable":sys.executable,"platform":platform.platform(),"packages":dict(sorted((d.metadata["Name"],d.version) for d in importlib.metadata.distributions()))}))

AUDIT_KERNEL={"python": "3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]", "executable": "C:\\Users\\Miakhlov\\Documents\\GitHub\\Computational chemistry_course\\scripts\\environment\\.venv-chem\\Scripts\\python.exe", "platform": "Windows-10-10.0.19045-SP0", "packages": {"Jinja2": "3.1.6", "MarkupSafe": "3.0.4", "PyYAML": "6.0.3", "Pygments": "2.21.0", "Send2Trash": "2.2.0", "altair": "6.3.0", "anyio": "4.16.0", "argon2-cffi": "25.1.0", "argon2-cffi-bindings": "26.2.0", "arrow": "1.4.0", "asttokens": "3.0.2", "async-lru": "2.4.0", "attrs": "26.1.0", "babel": "2.18.0", "beautifulsoup4": "4.15.0", "bleach": "6.4.0", "certifi": "2026.10.23", "cffi": "2.1.1", "charset-normalizer": "3.5.2", "click": "8.5.1", "cloudpickle": "3.1.3", "colorama": "0.4.6", "comm": "0.2.3", "contourpy": "1.4.0", "cycler": "0.12.1", "debugpy": "1.8.22", "defusedxml": "0.7.1", "executing": "2.3.0", "fastjsonschema": "2.22.3", "fonttools": "4.68.0", "fqdn": "1.6.0", "h11": "0.16.0

# Разбор A. Реальная UHPLC–MS ксантинов: разделение и устойчивость

Работаем с опубликованными данными [DOI 10.1155/2019/2926580](https://doi.org/10.1155/2019/2926580), Tables 2/4, Creative Commons Attribution (версия лицензии в XML не указана). Объекты A — теобромин, теофиллин и кофеин, не истрадефиллин. Матрица — чай; перенос на фармацевтическую композицию не подтверждён.

Условия Table 2: Nucleodur C18 polar Tec, 50×2 mm, 1.8 μm; ACN/water 10:90 с 1% муравьиной кислоты, 0.5 mL/min, 70 °C, инъекция 5 μL, ESI+ SIM. Опубликованные t_R трёх соединений — 14, 19, 28 s. Это не сырые хроматограммы; Gaussian-пики не дорисовываем как измерения.

In [2]:
from pathlib import Path
import sys,json,csv,math
import numpy as np
from scipy import stats
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
OUT=workspace('M10-L01','case')
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=150,bbox_inches='tight');display(fig);plt.close(fig)


In [3]:
with (DATA/'xanthine_uhplc_ruggedness.csv').open(encoding='utf-8',newline='') as f:rows=list(csv.DictReader(f))
t=np.array([float(r['retention_s']) for r in rows])
rsd=np.array([float(r['RSD_percent']) for r in rows])
n=np.array([int(r['n']) for r in rows])


Table 4 подписана «t_R ± %RSD (n=3)». Число после ± — относительная величина, не абсолютное SD. Переведём SD=t_R·%RSD/100, затем условный Student t-интервал среднего. Независимость и нормальность трёх повторов не подтверждены raw data; средние округлены до целых секунд, поэтому интервалы не восстанавливают исходную точность.

In [4]:
sd=t*rsd/100
half=stats.t.ppf(.975,n-1)*sd/np.sqrt(n)
change=100*(t-t[0])/t[0]
display([{'condition':r['condition'],'tR_s':float(mean),'derived_SD_s':float(s),
          'conditional_mean_CI95_s':[float(mean-h),float(mean+h)],'change_percent':float(c)}
         for r,mean,s,h,c in zip(rows,t,sd,half,change)])
fig,ax=plt.subplots(figsize=(9,4))
ax.errorbar(np.arange(len(t)),t,yerr=half,fmt='o');ax.axhline(t[0],color='gray')
ax.set_xticks(np.arange(len(t)),[r['condition'] for r in rows],rotation=35,ha='right')
ax.set(ylabel='Caffeine retention time / s',title='Published means; conditional intervals from rounded summary data')
publish_plot(fig,'ruggedness.png')


[{'condition': 'Optimum conditions',
  'tR_s': 28.0,
  'derived_SD_s': 0.1736,
  'conditional_mean_CI95_s': [27.568753693240144, 28.431246306759856],
  'change_percent': 0.0},
 {'condition': '0.55\u2009mL/min',
  'tR_s': 23.0,
  'derived_SD_s': 0.1127,
  'conditional_mean_CI95_s': [22.72003767988574, 23.27996232011426],
  'change_percent': -17.857142857142858},
 {'condition': '0.45\u2009mL/min',
  'tR_s': 35.0,
  'derived_SD_s': 0.14,
  'conditional_mean_CI95_s': [34.652220720354954, 35.347779279645046],
  'change_percent': 25.0},
 {'condition': 'ACN: H2O (15\u2009:\u2009\u200385, v/v)',
  'tR_s': 24.0,
  'derived_SD_s': 0.07919999999999999,
  'conditional_mean_CI95_s': [23.803256293229374, 24.196743706770626],
  'change_percent': -14.285714285714286},
 {'condition': 'ACN: H2O (5\u2009:\u2009\u200395, v/v)',
  'tR_s': 33.0,
  'derived_SD_s': 0.1848,
  'conditional_mean_CI95_s': [32.54093135086854, 33.45906864913146],
  'change_percent': 17.857142857142858},
 {'condition': '75°C',
  'tR

<Figure size 900x400 with 1 Axes>

Проверим простую альтернативу: при неизменных k′ и объёмах t_R∝1/F. Это диагностическая физическая аппроксимация, не новый fit с доказанной применимостью.

In [5]:
for i in [1,2]:
    flow=float(rows[i]['flow_mL_min']);ideal=t[0]*.5/flow
    print({'flow_mL_min':flow,'published_tR_s':float(t[i]),'inverse_flow_approximation_s':float(ideal),'difference_s':float(t[i]-ideal)})
retention=np.array([14.,19.,28.])
print('Published adjacent retention gaps / s:',np.diff(retention))
result={'kind':'calculated','scope':'published UHPLC summary reanalysis','conditional_SD_s':sd.tolist(),
        'relative_change_percent':change.tolist(),'measured_Rs':'NOT_CALCULABLE: numerical baseline widths unavailable',
        'new_injections':'NOT_RUN','scientific_review':'NOT_RUN'}


{'flow_mL_min': 0.55, 'published_tR_s': 23.0, 'inverse_flow_approximation_s': 25.454545454545453, 'difference_s': -2.4545454545454533}
{'flow_mL_min': 0.45, 'published_tR_s': 35.0, 'inverse_flow_approximation_s': 31.11111111111111, 'difference_s': 3.8888888888888893}
Published adjacent retention gaps / s: [5. 9.]


Небольшой %RSD внутри условия не означает отсутствия сдвига между условиями: изменение t_R достигает 25%. Без площади, ширин, асимметрии, матричных/blank-контролей и независимых приготовлений нельзя подтвердить полную устойчивость количественного метода. Разность t_R не равна Rs. Сформулируйте, какие измерения нужны, прежде чем принимать авторский общий вывод об устойчивости.

In [6]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
print(append_record(init_dossier(),'M10-L01','A',result,ROOT,inputs=['practicuum/shared/data/xanthine_uhplc_ruggedness.csv', 'practicuum/shared/data/xanthine_uhplc_provenance.json', 'practicuum/shared/data/source-responses/xanthine_uhplc.xml', 'practicuum/shared/data/atorvastatin_hplc_table1.csv', 'practicuum/shared/data/hplc_provenance.json']))


C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\practicuum\evidence\20261119-main-unification\notebooks\workspace\my_research\evidence\M10-L01\20261119T123328622555Z-A-e1c7d79ff83b44fd843d7ba784d19948.json
